# G4 · ReNeg on top of TINS: the final run (checklist 3.2b)

The probe P4 passed its pre-registered test: replayed on TINS's per-image scores, ReNeg-balanced on top of TINS
beat TINS on near-OOD, far-OOD and Four-OOD in every stream order. This notebook produces those numbers the
official way, on the exact features and in one pass per stream: TINS and TANL run once per batch, and several
ReNeg heads read their scores (ReNeg never changes its base, so each head scores exactly as it would alone).

* Labelled images: the same 5 per class as P4 (OpenOOD's ImageNet ID-val split). They are TINS's prototypes and
  enter every ReNeg head's ID image prototypes at each reset ("5-shot").
* Protocols: OpenOOD v1.5, and Four-OOD on the 45k ID images that are not labelled images (`four_ood_45k`, the
  same streams as G3's few-shot rows). Three stream orders.
* Scores saved per image for every head, so nothing has to run again for a new table.

**Time:** about 2–3 hours on an RTX 4060-class GPU (TINS dominates; P4 took 1 h 48 min). Every stream saves when it
finishes; after a stop, **Run all** again and finished streams are skipped.

**Output:** `MyDrive/ReNeg/transfers/g4_tins/`: `g4_runs.csv`, `g4_summary.csv` and `g4_scores_*.zip` (attach all).

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True)
ctx = start("G4_reneg_on_tins", need_gpu=True)
log = ctx.log
import json, time, glob, zipfile, shutil
import numpy as np, pandas as pd, torch
from oodlab.report import find_cache
from oodlab.runner import Runner, Protocol
from oodlab.metrics import openood_metrics
from oodlab.clipwrap import load_clip
from oodlab.methods import TANL, TANLConfig
from reneg.core import ReNegConfig
from reneg.multi import HeadSpec, MultiBase
from reneg.packs import load_kg_pool
from reneg.tins import TINSConfig, TINSFeat, build_init_candidates, build_static_negatives
from reneg.transport import l2n
cache = find_cache(ctx, "ViT-B/16")
bank = cache.load_textbank()
runner = Runner(cache, device=ctx.device, logger=log)
OUT = f"{DRIVE_ROOT}/transfers/g4_tins"
SC = os.path.join(OUT, "g4_scores")
os.makedirs(SC, exist_ok=True)
SEEDS = [0, 1, 2]
WEIGHTS = f"{DRIVE_ROOT}/weights"
model = load_clip("ViT-B/16", device=ctx.device, input_roots=ctx.input_roots + [WEIGHTS], download_root=WEIGHTS,
                  random_init=os.environ.get("RENEG_FAKE_GEN") == "1")
REAL = (hasattr(model, "token_embedding") and hasattr(TANL, "selected_T")
        and all(s in cache.available() for s in ("imagenet_val", "imagenet_val_all")))
print("device:", ctx.device, "| real oodlab and CLIP text encoder:", REAL)

## Labelled images, TINS's negatives (reused from P4), the KG pool and the 45k Four-OOD protocol

In [ ]:
if REAL:
    v5 = runner.get("imagenet_val")
    X5, y5 = l2n(v5.feats.float()), torch.as_tensor(np.asarray(v5.labels)).long()
    C = len(bank.id_names)
    cnt = torch.bincount(y5, minlength=C)
    proto = l2n(torch.zeros(C, X5.shape[1]).index_add_(0, y5, X5))
    if (cnt == 0).any():
        print(f"WARNING: {int((cnt == 0).sum())} classes have no labelled image: their text embedding is used")
        proto[cnt == 0] = l2n(bank.id_text.float())[cnt == 0]
    prior = (v5.feats, np.asarray(v5.labels))
    static_path = os.path.join(OUT, "tins_static.pt")
    p4_static = f"{DRIVE_ROOT}/transfers/p4_tins/tins_static.pt"
    if not os.path.isfile(static_path) and os.path.isfile(p4_static):
        shutil.copy2(p4_static, static_path)
    if os.path.isfile(static_path):
        st = torch.load(static_path, weights_only=False)
        assert torch.allclose(st["proto"], proto, atol=1e-5), "the labelled images differ from P4's"
    else:
        neg, words = build_static_negatives(model, bank.id_text.float(), proto, n=2000,
                                            positive_labels=list(bank.id_names), log=print)
        st = {"neg": neg, "words": words, "cands": build_init_candidates(model, words, proto), "proto": proto}
        torch.save(st, static_path)
    print(len(st["words"]), "TINS static negatives;", len(X5), "labelled ID images")
    # Four-OOD without the labelled images (as G3's few-shot rows)
    va = runner.get("imagenet_val_all")
    A, B = l2n(va.feats.float().to(ctx.device)), X5.to(ctx.device)
    shot = torch.zeros(len(va), dtype=torch.bool)
    for s in range(0, len(va), 4096):
        shot[s:s + 4096] = ((A[s:s + 4096] @ B.T).max(1).values > 0.999).cpu()
    shot = shot.numpy()
    np.save(os.path.join(OUT, "shot_mask.npy"), shot)
    del A, B
    runner.extra_sets["imagenet_val_45k"] = va.subset(np.nonzero(~shot)[0], name="imagenet_val_45k")
    P45 = Protocol("four_ood_45k", "imagenet_val_45k", {"four45": ["inaturalist", "sun", "places", "textures_all"]},
                   "Four-OOD without OpenOOD's 5k ID-val images (they are the labelled images)")
    print("labelled images found in the 50k Four-OOD ID set:", int(shot.sum()))
    # KG pool (as G3)
    kg = load_kg_pool()["pool"]
    z = np.load(f"{DRIVE_ROOT}/transfers/analysis_pack2/reneg_pool.npz")
    assert [r["name"] for r in kg] == list(z["pool_names"]), "pool file and kg_pool.json disagree"
    pool_text = l2n(torch.tensor(z["pool_text"]).float())
    pool_cluster = np.array([r["cluster"] for r in kg])
    blind = ~np.array([bool(r["in_ssb_hard"]) or bool(r["in_ninco"]) for r in kg])
    pools = {"full": (pool_text, pool_cluster),
             "blind": (pool_text[torch.as_tensor(np.nonzero(blind)[0])], pool_cluster[blind])}
    BASE = dict(n_min=1, pool_frac=0.0, pool_in_text=False, id_admit="softmax+base")
    MODES = {"balanced": ReNegConfig(**BASE, image_mode="kg_clip", vote=True), "max": ReNegConfig(**BASE),
             "safe": ReNegConfig(**BASE, image_mode="kg_clip_caught", vote=True)}
    HEADS = [HeadSpec("balanced on tins", "tins", MODES["balanced"]),
             HeadSpec("max on tins", "tins", MODES["max"]),
             HeadSpec("safe on tins", "tins", MODES["safe"]),
             HeadSpec("balanced on tins, full pool", "tins", MODES["balanced"], pool="full"),
             HeadSpec("balanced on tins, no labelled images", "tins", MODES["balanced"], prior=False),
             HeadSpec("balanced on tanl+tins", "tanl+tins", MODES["balanced"]),
             HeadSpec("max on tanl+tins", "tanl+tins", MODES["max"]),
             HeadSpec("safe on tanl+tins", "tanl+tins", MODES["safe"]),
             HeadSpec("balanced on tanl", "tanl", MODES["balanced"])]
    NAMES = ["tins", "tanl", "tanl+tins"] + [h.name for h in HEADS]
    print(len(kg), "KG names,", int(blind.sum()), "in the blind pool;", len(HEADS), "ReNeg heads")
else:
    print("test stub of oodlab (no TANL / CLIP text encoder): the G4 cells are skipped")

## Timing check on one stream (NINCO, stream order 0) and an estimate for the whole run

In [ ]:
def make_multi():
    tins = TINSFeat(model, bank.id_text.float(), st["proto"], st["neg"], st["cands"], TINSConfig())
    tanl = TANL(bank.id_text, bank.corpus_text, bank.noise_feats, TANLConfig.paper().with_(record=True),
                device=ctx.device, n_neglabel=int(bank.n_selected))
    return MultiBase(tins, tanl, bank.id_text, pools, HEADS, primary="balanced on tins", device=ctx.device,
                     id_prior=prior)

class ScoreHook:
    """Saves every score (bases and heads) per image in stream order, with the labels and stream rows."""
    def __init__(self, path):
        self.path, self.parts = path, []
    def on_reset(self, method, seg):
        self.parts = []
    def on_step(self, method, seg, start, end, out, stream):
        self.parts.append((start, {k: v.numpy() for k, v in out.info["scores"].items()}))
    def on_end(self, method, stream):
        n = len(stream)
        arr = np.zeros((len(NAMES), n), np.float32)
        for s, d in self.parts:
            for i, k in enumerate(NAMES):
                arr[i, s:s + len(d[k])] = d[k]
        tmp = self.path + ".tmp.npz"
        np.savez_compressed(tmp, scores=arr, names=np.asarray(NAMES), is_ood=(np.asarray(stream.labels) == -1),
                            rows=stream.rows.astype(np.int32), n_inv=method.tins.n_inv, t_inv=method.tins.t_inv)
        os.replace(tmp, self.path)

def score_path(protocol, ds, seed):
    return os.path.join(SC, f"{protocol}__{ds}__s{seed}.npz")

runs_path = os.path.join(OUT, "g4_runs.csv")

def rows_from(path, protocol, ds, seed, seconds=None):
    z = np.load(path)
    lab = np.where(z["is_ood"], -1, 0)
    out = []
    for i, k in enumerate(z["names"]):
        m = openood_metrics(z["scores"][i], lab)
        out.append({"label": str(k), "protocol": protocol, "dataset": ds, "seed": seed, "fpr95": m["fpr95"],
                    "fpr95_idpos": m["fpr95_idpos"], "auroc": m["auroc"], "n_id": m["n_id"], "n_ood": m["n_ood"],
                    "n_inverted": int(z["n_inv"]), "inversion_seconds": float(z["t_inv"]), "stream_seconds": seconds})
    return out

def run_one(protocol, ds, seed):
    global done
    path = score_path(protocol, ds, seed)
    if os.path.isfile(path):
        return None
    t0 = time.time()
    runner.evaluate(make_multi(), P45 if protocol == "four_ood_45k" else protocol, seeds=[seed], datasets=[ds],
                    config="g4", hooks_factory=lambda name, sd: [ScoreHook(score_path(protocol, name, sd))])
    rows = pd.DataFrame(rows_from(path, protocol, ds, seed, time.time() - t0))
    done = pd.concat([done, rows], ignore_index=True)
    done.to_csv(runs_path, index=False)
    r = rows.set_index("label")
    print(f"{protocol} {ds} s{seed}: TINS {r.loc['tins', 'fpr95_idpos']:.2f} | ReNeg-balanced on TINS "
          f"{r.loc['balanced on tins', 'fpr95_idpos']:.2f} | on TANL+TINS {r.loc['balanced on tanl+tins', 'fpr95_idpos']:.2f}"
          f" (FPR95, ID positive); {(time.time() - t0) / 60:.1f} min", flush=True)
    return rows

SETS = {"openood_v15": (["ssb_hard", "ninco", "inaturalist", "textures", "openimage_o"], "imagenet_test"),
        "four_ood_45k": (["inaturalist", "sun", "places", "textures_all"], "imagenet_val_45k")}
done = pd.read_csv(runs_path) if os.path.isfile(runs_path) else pd.DataFrame()
if REAL:
    run_one("openood_v15", "ninco", 0)
    sel = done[(done.protocol == "openood_v15") & (done.dataset == "ninco") & (done.seed == 0)]
    if len(sel) and pd.notna(sel.stream_seconds.iloc[0]):
        per = float(sel.stream_seconds.iloc[0]) / len(np.load(score_path("openood_v15", "ninco", 0))["is_ood"])
        n_img = sum(len(runner.get(d)) + len(runner.get(SETS[p][1])) for p in SETS for d in SETS[p][0])
        print(f"rough estimate: {len(SEEDS) * n_img * per / 3600:.1f} hours for everything (a lower bound: NINCO "
              f"has few OOD images; P4, TINS alone, took 1 h 48 min on an RTX 4060)")

## Run every stream (stream order 0 for all sets first, then 1 and 2; saved per stream)

In [ ]:
if REAL:
    for seed in SEEDS:
        for protocol in SETS:
            for ds in SETS[protocol][0]:
                run_one(protocol, ds, seed)

## Summary (both FPR95 conventions), checks, and the files to attach

Pass rule fixed before P4's numbers: ReNeg-balanced on TINS (blind pool, 5-shot) beats TINS on near-OOD, far-OOD
and Four-OOD in ID-positive FPR95 (mean of three orders), and on near and far in every order. `balanced on tanl`
should equal G3's `reneg_balanced_blind_5shot` on the same streams (same code, same seeds).

In [ ]:
def TRACES():
    return sorted(f for f in glob.glob(os.path.join(SC, "*.npz")) if not f.endswith(".tmp.npz"))

allrows = []
for f in TRACES():
    protocol, ds, s = os.path.basename(f)[:-4].split("__")
    allrows += rows_from(f, protocol, ds, int(s[1:]))
if allrows:
    df = pd.DataFrame(allrows)
    GROUPS = {"near": ("openood_v15", ["ssb_hard", "ninco"]),
              "far": ("openood_v15", ["inaturalist", "textures", "openimage_o"]),
              "Four-OOD 45k": ("four_ood_45k", ["inaturalist", "sun", "places", "textures_all"])}

    def gseed(lab, prot, sets_, col):
        g = df[(df.label == lab) & (df.protocol == prot) & df.dataset.isin(sets_)]
        n = g.groupby("seed").size()
        return g[g.seed.isin(n[n == len(sets_)].index)].groupby("seed")[col].mean()

    rows = []
    for lab in NAMES:
        row = {"method": lab}
        for gname, (prot, sets_) in GROUPS.items():
            for col, short in (("fpr95_idpos", "FPR95 ID+"), ("fpr95", "FPR95 OOD+"), ("auroc", "AUROC")):
                v = gseed(lab, prot, sets_, col)
                if len(v):
                    row[f"{gname} {short}"] = round(v.mean(), 2)
        rows.append(row)
    summary = pd.DataFrame(rows)
    summary.to_csv(os.path.join(OUT, "g4_summary.csv"), index=False)
    pd.set_option("display.width", 250)
    print(summary.to_string(index=False))
    print("\npass rule (ID-positive FPR95, ReNeg-balanced on TINS minus TINS, per order):")
    ok = True
    for gname, (prot, sets_) in GROUPS.items():
        d = (gseed("balanced on tins", prot, sets_, "fpr95_idpos") - gseed("tins", prot, sets_, "fpr95_idpos")).dropna()
        good = len(d) == len(SEEDS) and d.mean() < 0 and (gname == "Four-OOD 45k" or (d < 0).all())
        ok &= good
        print(f"  {gname}: mean {d.mean():+.2f}, per order {np.round(d.values, 2).tolist()} -> {'PASS' if good else 'FAIL'}")
    print("  overall:", "PASS" if ok else "FAIL (or not finished)")
    g3 = f"{DRIVE_ROOT}/transfers/g3_final_b16/g3_runs.csv"
    if os.path.isfile(g3):
        t = pd.read_csv(g3)
        for mine, theirs in (("balanced on tanl", "reneg_balanced_blind_5shot"), ("tanl", "tanl")):
            a = df[df.label == mine].merge(t[t.label == theirs], on=["protocol", "dataset", "seed"], suffixes=("", "_g3"))
            if len(a):
                print(f"check vs G3 {theirs}: {len(a)} streams, largest FPR95 difference "
                      f"{(a.fpr95 - a.fpr95_g3).abs().max():.3f} (expect a few tenths at most: G3 used the "
                      f"5,002 matched copies inside the 50k set as its labelled images)")
parts, cur, size = [], [], 0
for f in TRACES():
    if cur and size + os.path.getsize(f) > 22e6:
        parts.append(cur); cur, size = [], 0
    cur.append(f); size += os.path.getsize(f)
if cur:
    parts.append(cur)
for old in glob.glob(os.path.join(OUT, "g4_scores_*.zip")):
    os.remove(old)
for k, fs in enumerate(parts, 1):
    with zipfile.ZipFile(os.path.join(OUT, f"g4_scores_{k}.zip"), "w") as zf:
        for f in fs:
            zf.write(f, os.path.basename(f))
print("\nresult files:")
for f in [runs_path, os.path.join(OUT, "g4_summary.csv")] + sorted(glob.glob(os.path.join(OUT, "g4_scores_*.zip"))):
    if os.path.isfile(f):
        print("  ", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")
finish(ctx, {"streams": len(TRACES())})

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")